# نهایی — Supply Chain R&D Portfolio Analysis

**Input:** `clean_corpus_checkpoint/clean.csv` (exactly 7,747 records, preserved order)  
**Embeddings:** `paraphrase-multilingual-mpnet-base-v2`, normalized, saved alongside the CSV  
**Output:** `clean_corpus_checkpoint/pipeline_outputs_final/`

This notebook starts **after** preprocessing. It does not load the ZIP/Excel sources, deduplicate, translate, normalize or remove any additional records. There is **no medical filtering step** in this notebook. The contents of `clean.csv` already reflect whatever filtering occurred when it was created; this notebook cannot restore previously excluded records.

The downstream analysis (embedding, k selection, clustering, keywords, lifecycle, portfolio metrics, figures and report) follows the uploaded `Grants_Persian_Paper.ipynb` notebook. The original data-driven k selection (5–18) is retained; optionally set `FIXED_K=11` below if you need k=11 specifically.


In [ ]:
%pip -q install sentence-transformers scikit-learn scipy matplotlib seaborn pandas numpy

In [ ]:
# CONFIGURATION — paths match the clean.csv saved by the previous cleaning cell
from pathlib import Path
import os, json, re, time, hashlib, tempfile, gc, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/Academic/Thesis/Grants Persian Paper')
CHECKPOINT_DIR = PROJECT_DIR / 'clean_corpus_checkpoint'
CSV_PATH = CHECKPOINT_DIR / 'clean.csv'
PREPROCESSING_MANIFEST_PATH = CHECKPOINT_DIR / 'preprocessing_manifest.json'
EMBEDDING_PATH = CHECKPOINT_DIR / 'embeddings.npy'
EMBEDDING_MANIFEST_PATH = CHECKPOINT_DIR / 'embedding_manifest.json'
OUTPUT_DIR = str(CHECKPOINT_DIR / 'pipeline_outputs_final')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

EXPECTED_N = 7747
MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
FIXED_K = None  # Set to 11 to force k=11; None keeps original k-selection procedure.
FORCE_REBUILD_EMBEDDINGS = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Input: {CSV_PATH}')
print(f'Output: {OUTPUT_DIR}')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def sha256_file(path, chunk_bytes=4*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_bytes), b''):
            h.update(chunk)
    return h.hexdigest()

def ordered_text_sha256(texts):
    h = hashlib.sha256()
    for text in texts:
        payload = text.encode('utf-8')
        h.update(len(payload).to_bytes(8, 'big'))
        h.update(payload)
    return h.hexdigest()


In [ ]:
# STEP 1 — Read only the saved clean CSV (no further filtering or sampling)
if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f'Cannot find {CSV_PATH}. First run the prior cleaning/export cell that writes clean.csv.'
    )

# Read textual IDs and metadata as strings, preserve CSV row order.
df_raw = pd.read_csv(CSV_PATH, encoding='utf-8-sig', dtype=str, keep_default_na=False)
if len(df_raw) != EXPECTED_N:
    raise ValueError(f'Expected {EXPECTED_N:,} records in clean.csv; found {len(df_raw):,}.')
required_columns = {'clean_text', 'combined_text'}
missing_columns = required_columns - set(df_raw.columns)
if missing_columns:
    raise ValueError(f'Missing required columns in clean.csv: {sorted(missing_columns)}')
if not df_raw['clean_text'].map(lambda t: isinstance(t, str) and bool(t.strip())).all():
    raise ValueError('clean_text contains blank values. Input should be the saved clean.csv.')

# Keep all rows and preserve their exact original order.
n_before = len(df_raw)
texts = df_raw['clean_text'].tolist()
texts_sha256 = ordered_text_sha256(texts)

# Preserve column mapping from the cleaning manifest if available.
col_map = {}
if PREPROCESSING_MANIFEST_PATH.is_file():
    manifest = json.loads(PREPROCESSING_MANIFEST_PATH.read_text(encoding='utf-8'))
    col_map = {k: v for k, v in manifest.get('column_mapping', {}).items()
               if v in df_raw.columns}

# Fallback: use the original notebook's column-detection rules when necessary.
for col in df_raw.columns:
    cl = col.strip().lower()
    if any(k in cl for k in ['title', 'grant title', 'project title']):
        col_map.setdefault('title', col)
    elif any(k in cl for k in ['abstract', 'description', 'summary']):
        col_map.setdefault('abstract', col)
    elif any(k in cl for k in ['keyword', 'key word']):
        col_map.setdefault('keywords', col)
    elif any(k in cl for k in ['award year', 'start year', 'start date', 'year', 'date']):
        col_map.setdefault('year', col)
    elif any(k in cl for k in ['agency', 'funder', 'funding body', 'grant agency']):
        col_map.setdefault('agency', col)
    elif any(k in cl for k in ['grant id', 'grant number', 'award id', 'uid', 'accession']):
        col_map.setdefault('grant_id', col)
    elif any(k in cl for k in ['country', 'nation', 'region']):
        col_map.setdefault('country', col)
    elif any(k in cl for k in ['amount', 'budget', 'award amount', 'funding amount']):
        col_map.setdefault('amount', col)

def safe_get(row, key):
    col = col_map.get(key)
    if col and col in row.index:
        val = str(row[col]).strip()
        return val if val.lower() not in ('nan', 'none', '') else ''
    return ''

print(f'Loaded exactly {len(df_raw):,} cleaned rows from {CSV_PATH.name}')
print(f'Input CSV SHA256: {sha256_file(CSV_PATH)}')
print(f'Ordered clean_text SHA256: {texts_sha256}')
print(f'Columns: {len(df_raw.columns)} | mapping: {col_map}')
print('All existing clean_text and combined_text values retained; no preprocessing repeated.')


In [ ]:
# STEP 7 — Original multilingual MPNet embedding, saved beside clean.csv
from sentence_transformers import SentenceTransformer

embeddings = None
if EMBEDDING_PATH.is_file() and EMBEDDING_MANIFEST_PATH.is_file() and not FORCE_REBUILD_EMBEDDINGS:
    try:
        meta = json.loads(EMBEDDING_MANIFEST_PATH.read_text(encoding='utf-8'))
        cache_matches = (
            meta.get('model_name') == MODEL_NAME
            and meta.get('n_records') == EXPECTED_N
            and meta.get('ordered_clean_text_sha256') == texts_sha256
            and meta.get('normalize_embeddings') is True
            and meta.get('embedding_file_sha256') == sha256_file(EMBEDDING_PATH)
        )
        if cache_matches:
            candidate = np.load(EMBEDDING_PATH, allow_pickle=False)
            if (candidate.ndim == 2 and candidate.shape[0] == EXPECTED_N
                    and candidate.shape[1] >= 2 and np.isfinite(candidate).all()
                    and np.allclose(np.linalg.norm(candidate, axis=1), 1., atol=1e-3)):
                embeddings = candidate.astype(np.float32, copy=False)
                print('Using verified embedding cache:', EMBEDDING_PATH)
    except (OSError, ValueError, KeyError, TypeError, json.JSONDecodeError) as exc:
        print('Existing embedding cache invalid:', exc)

if embeddings is None:
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        BATCH_SIZE = 256 if vram_gb >= 30 else 128 if vram_gb >= 14 else 64
    else:
        BATCH_SIZE = 32
    print(f'Building embeddings using {MODEL_NAME}; {DEVICE=}, {BATCH_SIZE=}')
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    t0 = time.time()
    embeddings = model.encode(
        texts, batch_size=BATCH_SIZE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True, device=DEVICE,
    )
    embeddings = np.asarray(embeddings, dtype=np.float32)
    if (embeddings.ndim != 2 or embeddings.shape[0] != EXPECTED_N
            or embeddings.shape[1] < 2 or not np.isfinite(embeddings).all()
            or not np.allclose(np.linalg.norm(embeddings, axis=1), 1., atol=1e-3)):
        raise RuntimeError(f'Unexpected embeddings: {embeddings.shape}')

    # Atomic overwrite; save manifest only after verifying the new file.
    with tempfile.NamedTemporaryFile('wb', suffix='.npy', dir=CHECKPOINT_DIR, delete=False) as f:
        tmp_path = Path(f.name)
        np.save(f, embeddings, allow_pickle=False)
    os.replace(tmp_path, EMBEDDING_PATH)
    metadata = {
        'model_name': MODEL_NAME,
        'text_column': 'clean_text',
        'n_records': EXPECTED_N,
        'ordered_clean_text_sha256': texts_sha256,
        'csv_sha256': sha256_file(CSV_PATH),
        'embedding_filename': EMBEDDING_PATH.name,
        'embedding_file_sha256': sha256_file(EMBEDDING_PATH),
        'embedding_dimensions': int(embeddings.shape[1]),
        'normalize_embeddings': True,
        'convert_to_numpy': True,
        'batch_size': BATCH_SIZE,
        'device': DEVICE,
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'elapsed_seconds': time.time() - t0,
    }
    with tempfile.NamedTemporaryFile('w', suffix='.json', dir=CHECKPOINT_DIR,
                                     encoding='utf-8', delete=False) as f:
        tmp_manifest = Path(f.name)
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    os.replace(tmp_manifest, EMBEDDING_MANIFEST_PATH)
    del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print('New embeddings generated and saved:', EMBEDDING_PATH)

assert len(df_raw) == len(embeddings) == EXPECTED_N
print('Embedding dimensions:', embeddings.shape)


In [ ]:
# ── STEP 8: Clustering Optimization ──────────────────────────
print("\n" + "=" * 60)
print("STEP 8 — Clustering optimization (k = 5 to 18)")
print("=" * 60)

from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import (silhouette_score,
                              davies_bouldin_score,
                              calinski_harabasz_score)

def balance_ratio(labels):
    counts = np.bincount(labels)
    return counts.min() / counts.max()

USE_MINIBATCH = len(df_raw) > 10000
print(f"Using {'MiniBatchKMeans' if USE_MINIBATCH else 'KMeans'}")

results = []
for k in range(5, 19):  # Adjusted range for cleaner data
    if USE_MINIBATCH:
        km = MiniBatchKMeans(n_clusters=k, init="k-means++", n_init=10,
                              random_state=42, max_iter=300, batch_size=2048)
    else:
        km = KMeans(n_clusters=k, init="k-means++", n_init=10,
                    random_state=42, max_iter=300)

    labels = km.fit_predict(embeddings)
    sil = silhouette_score(embeddings, labels,
                            sample_size=min(5000, len(df_raw)), random_state=42)
    dbi = davies_bouldin_score(embeddings, labels)
    chi = calinski_harabasz_score(embeddings, labels)
    br  = balance_ratio(labels)
    results.append({"k": k, "silhouette": sil, "davies_bouldin": dbi,
                    "calinski_harabasz": chi, "balance_ratio": br})
    print(f"  k={k:2d} | Sil={sil:.4f} | DBI={dbi:.3f} | CHI={chi:.1f} | BR={br:.3f}")

df_eval = pd.DataFrame(results)

def norm_col(s, invert=False):
    mn, mx = s.min(), s.max()
    n = (s - mn) / (mx - mn + 1e-9)
    return 1 - n if invert else n

# Balanced scoring without artificial k preference
df_eval["score"] = (
    3.5 * norm_col(df_eval["silhouette"]) +
    2.5 * norm_col(df_eval["davies_bouldin"], invert=True) +
    2.0 * norm_col(df_eval["calinski_harabasz"]) +
    2.0 * norm_col(df_eval["balance_ratio"])
)

best_row = df_eval.loc[df_eval["score"].idxmax()]
BEST_K = int(best_row["k"])
if FIXED_K is not None:
    if int(FIXED_K) not in df_eval['k'].values:
        raise ValueError(f'FIXED_K must be in k=5..18, got {FIXED_K}')
    BEST_K = int(FIXED_K)
    best_row = df_eval.loc[df_eval['k'] == BEST_K].iloc[0]
    print(f'Fixed k override selected: {BEST_K}')

print(f"\nFull evaluation table:")
print(df_eval.to_string(index=False))
print(f"\nSELECTED k = {BEST_K} (composite score = {best_row['score']:.4f})")
df_eval.to_csv(os.path.join(OUTPUT_DIR, "clustering_evaluation.csv"), index=False)



In [ ]:
# ── STEP 9: Final Clustering ──────────────────────────────────
print("\n" + "=" * 60)
print(f"STEP 9 — Final clustering: k = {BEST_K}")
print("=" * 60)

if USE_MINIBATCH:
    km_final = MiniBatchKMeans(n_clusters=BEST_K, init="k-means++", n_init=20,
                                random_state=42, max_iter=500, batch_size=2048)
else:
    km_final = KMeans(n_clusters=BEST_K, init="k-means++", n_init=20,
                       random_state=42, max_iter=500)

df_raw["cluster"] = km_final.fit_predict(embeddings)

cluster_sizes = df_raw["cluster"].value_counts().sort_index()
print("Cluster size distribution:")
print(cluster_sizes.to_string())
print(f"\nMean: {cluster_sizes.mean():.1f} | Std: {cluster_sizes.std():.1f} | "
      f"Min: {cluster_sizes.min()} | Max: {cluster_sizes.max()}")


# Keep labels aligned with saved CSV rows; never overwrite clean.csv.
assert len(df_raw) == EXPECTED_N
clustered_path = Path(OUTPUT_DIR) / 'clustered_grants.csv'
df_raw.to_csv(clustered_path, index=False, encoding='utf-8-sig')
print(f'Clustered records saved: {clustered_path}')
if BEST_K == 11:
    np.save(CHECKPOINT_DIR / 'k11_reference_labels.npy',
            df_raw['cluster'].to_numpy(dtype=int), allow_pickle=False)
    print('k=11 reference labels saved beside embeddings.')


In [ ]:
# ── STEP 10: Advanced Keyword Extraction ─────────────────────
print("\n" + "=" * 60)
print("STEP 10 — Advanced keyword extraction per cluster")
print("=" * 60)

from sklearn.feature_extraction.text import TfidfVectorizer

def extract_cluster_keywords(df, cluster_col="cluster", text_col="clean_text", top_n=15):
    cluster_ids = sorted(df[cluster_col].unique())
    cluster_docs = []
    for c in cluster_ids:
        texts = df[df[cluster_col] == c][text_col].tolist()
        cluster_docs.append(" ".join(texts))

    tfidf = TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.85,
        sublinear_tf=True
    )
    tfidf_matrix = tfidf.fit_transform(cluster_docs)
    feature_names = np.array(tfidf.get_feature_names_out())

    cluster_keywords = {}
    for idx, c in enumerate(cluster_ids):
        scores = tfidf_matrix[idx].toarray().flatten()
        top_indices = scores.argsort()[::-1][:top_n]
        keywords = [(feature_names[i], round(float(scores[i]), 4))
                    for i in top_indices if scores[i] > 0]
        cluster_keywords[c] = keywords

    return cluster_keywords

cluster_keywords = extract_cluster_keywords(df_raw, top_n=15)

print("Top 15 TF-IDF keywords per cluster:")
for c in sorted(cluster_keywords.keys()):
    kw_str = ", ".join(f"{w}({s:.3f})" for w, s in cluster_keywords[c][:10])
    print(f"  C{c:02d} (n={cluster_sizes[c]:5d}): {kw_str}")



In [ ]:
# ── STEP 11: Year Extraction ──────────────────────────────────
print("\n" + "=" * 60)
print("STEP 11 — Year extraction")
print("=" * 60)

year_col = col_map.get("year")

def extract_year(val):
    if pd.isna(val):
        return None
    m = re.search(r"(19|20)\d{2}", str(val))
    return int(m.group()) if m else None

if year_col:
    df_raw["year_parsed"] = df_raw[year_col].apply(extract_year)
    print(f"Year extracted from: '{year_col}'")
else:
    df_raw["year_parsed"] = None
    print("No year column mapped. Scanning all columns...")
    for col in df_raw.columns:
        if col in ["source_file", "combined_text", "clean_text", "lang", "cluster"]:
            continue
        if df_raw["year_parsed"].isna().all():
            candidate = df_raw[col].apply(extract_year)
            if candidate.notna().sum() > len(df_raw) * 0.3:
                df_raw["year_parsed"] = candidate
                print(f"  Year found in column: '{col}'")
                break

year_valid = df_raw["year_parsed"].dropna()
print(f"Records with valid year: {len(year_valid)} ({len(year_valid)/len(df_raw)*100:.1f}%)")

if len(year_valid) > 0:
    print(f"Year range: {int(year_valid.min())} to {int(year_valid.max())}")
    annual = year_valid.astype(int).value_counts().sort_index()
    print("Annual distribution:")
    print(annual.to_string())
else:
    annual = pd.Series(dtype=int)
    print("WARNING: No valid years found.")



In [ ]:
# ── STEP 12: Lifecycle Classification (REVISED) ──────────────
print("\n" + "=" * 60)
print("STEP 12 — Lifecycle classification (REVISED THRESHOLDS)")
print("=" * 60)

CURRENT_YEAR = 2024
RECENT_WINDOW = [2020, 2021, 2022, 2023, 2024]  # 5-year window
lifecycle_results = []

for c in sorted(df_raw["cluster"].unique()):
    sub = df_raw[df_raw["cluster"] == c]
    n_total = len(sub)
    sub_year = sub["year_parsed"].dropna()

    if len(sub_year) == 0:
        lifecycle_results.append({
            "cluster": int(c), "n_grants": int(n_total),
            "peak_year": None, "recency_ratio": None,
            "growth_rate": None, "vitality_score": None,
            "lifecycle_stage": "Unknown"
        })
        continue

    year_counts = sub_year.value_counts().sort_index()
    peak_year = int(year_counts.idxmax())
    years_since_peak = CURRENT_YEAR - peak_year

    # Recency ratio (last 5 years)
    recent_count = int(sub_year[sub_year.isin(RECENT_WINDOW)].count())
    recency_ratio = recent_count / len(sub_year)

    # Growth rate (compare recent 3yr avg vs prior 3yr avg)
    recent_3yr = [2022, 2023, 2024]
    prior_3yr = [2019, 2020, 2021]
    recent_avg = sub_year[sub_year.isin(recent_3yr)].count() / 3
    prior_avg = sub_year[sub_year.isin(prior_3yr)].count() / 3
    growth_rate = ((recent_avg - prior_avg) / (prior_avg + 1)) if prior_avg > 0 else 0

    # Vitality score (composite)
    vitality = round((recency_ratio * 2.0) + (np.log1p(n_total) / 10) + (growth_rate * 0.5), 3)

    # REVISED CLASSIFICATION LOGIC
    if years_since_peak <= 2 and growth_rate > 0.3 and recency_ratio > 0.40:
        stage = "Emerging"
    elif years_since_peak <= 5 and growth_rate > 0.1 and recency_ratio > 0.30:
        stage = "Growth"
    elif years_since_peak <= 10 and recency_ratio > 0.15:
        stage = "Mature"
    else:
        stage = "Saturation"

    lifecycle_results.append({
        "cluster": int(c),
        "n_grants": int(n_total),
        "peak_year": int(peak_year),
        "years_since_peak": int(years_since_peak),
        "recency_ratio": round(float(recency_ratio), 3),
        "growth_rate": round(float(growth_rate), 3),
        "vitality_score": float(vitality),
        "lifecycle_stage": stage
    })
    print(f"  C{c:02d}: peak={peak_year} ({years_since_peak}y ago) | "
          f"recency={recency_ratio:.3f} | growth={growth_rate:.3f} | "
          f"vitality={vitality:.3f} | stage={stage}")

df_lifecycle = pd.DataFrame(lifecycle_results)
print(f"\nLifecycle distribution:")
print(df_lifecycle["lifecycle_stage"].value_counts().to_string())
df_lifecycle.to_csv(os.path.join(OUTPUT_DIR, "lifecycle_results.csv"), index=False)



In [ ]:
# ── STEP 13: Portfolio Balance Analysis ───────────────────────
print("\n" + "=" * 60)
print("STEP 13 — Portfolio Balance Analysis")
print("=" * 60)

stage_order  = ["Emerging", "Growth", "Mature", "Saturation"]
ideal_shares = {"Emerging": 0.20, "Growth": 0.35, "Mature": 0.30, "Saturation": 0.15}

stage_counts = df_lifecycle["lifecycle_stage"].value_counts()
total_clusters = len(df_lifecycle)

print("Stage distribution:")
for stage in stage_order:
    cnt = int(stage_counts.get(stage, 0))
    grant_share = int(df_lifecycle[df_lifecycle["lifecycle_stage"] == stage]["n_grants"].sum())
    print(f"  {stage:12s}: {cnt} clusters ({cnt/total_clusters*100:.1f}%) | "
          f"{grant_share} grants ({grant_share/len(df_raw)*100:.1f}%)")

grant_shares_hhi = df_lifecycle["n_grants"] / df_lifecycle["n_grants"].sum()
HHI = round(float((grant_shares_hhi ** 2).sum() * 10000), 1)
hhi_label = "diverse" if HHI < 1500 else "moderate" if HHI < 2500 else "concentrated"
print(f"\nHHI: {HHI} ({hhi_label})")

actual_grant_shares = {}
for stage in stage_order:
    total_g = int(df_lifecycle[df_lifecycle["lifecycle_stage"] == stage]["n_grants"].sum())
    actual_grant_shares[stage] = total_g / len(df_raw)

balance_score = 1 - sum(abs(actual_grant_shares.get(s, 0) - ideal_shares[s])
                        for s in stage_order) / 2

print(f"Portfolio Balance Score: {balance_score:.3f}")
print("Actual vs. Ideal:")
for s in stage_order:
    a = actual_grant_shares.get(s, 0)
    i = ideal_shares[s]
    direction = ("over-invested" if (a-i) > 0.05
                 else "under-invested" if (a-i) < -0.05
                 else "on-target")
    print(f"  {s:12s}: actual={a:.3f} | ideal={i:.3f} | gap={a-i:+.3f} | {direction}")

# Agency-stage matrix
agency_col = col_map.get("agency")
if agency_col:
    df_with_stage = df_raw.merge(
        df_lifecycle[["cluster", "lifecycle_stage"]], on="cluster", how="left"
    )
    top_agencies = df_with_stage[agency_col].value_counts().head(10).index.tolist()
    df_top = df_with_stage[df_with_stage[agency_col].isin(top_agencies)]
    agency_stage_matrix = pd.crosstab(df_top[agency_col], df_top["lifecycle_stage"])
    cols_present = [c for c in stage_order if c in agency_stage_matrix.columns]
    agency_stage_matrix = agency_stage_matrix[cols_present]
    print("\nAgency-Stage matrix (top 10 agencies):")
    print(agency_stage_matrix.to_string())
    agency_stage_matrix.to_csv(os.path.join(OUTPUT_DIR, "agency_stage_matrix.csv"))

portfolio_summary = {
    "HHI": float(HHI),
    "HHI_label": hhi_label,
    "balance_score": round(float(balance_score), 3),
    "actual_stage_shares": {k: round(float(v), 3) for k, v in actual_grant_shares.items()},
    "ideal_stage_shares": ideal_shares,
    "stage_cluster_counts": {k: int(v) for k, v in stage_counts.items()},
    "total_records": int(len(df_raw)),
    "total_clusters": int(total_clusters),
    "input_csv": str(CSV_PATH),
    "input_records": EXPECTED_N
}
with open(os.path.join(OUTPUT_DIR, "portfolio_analysis.json"), "w") as f:
    json.dump(portfolio_summary, f, indent=2)
print("Portfolio analysis saved.")



In [ ]:
# ── STEP 14: Agency Distribution ─────────────────────────────
print("\n" + "=" * 60)
print("STEP 14 — Funding agency distribution")
print("=" * 60)

if agency_col:
    agency_counts = df_raw[agency_col].value_counts().head(20)
    print("Top 20 agencies:")
    print(agency_counts.to_string())
    agency_df = agency_counts.reset_index()
    agency_df.columns = ["agency", "grant_count"]
    agency_df["pct"] = (agency_df["grant_count"] / len(df_raw) * 100).round(1)
    agency_df.to_csv(os.path.join(OUTPUT_DIR, "agency_distribution.csv"), index=False)
else:
    print("No agency column detected.")
    agency_df = pd.DataFrame()



In [ ]:
# ── STEP 15: Temporal Trajectories ───────────────────────────
print("\n" + "=" * 60)
print("STEP 15 — Temporal trajectories per cluster")
print("=" * 60)

trajectory_data = {}
for c in sorted(df_raw["cluster"].unique()):
    sub_year = df_raw[df_raw["cluster"] == c]["year_parsed"].dropna()
    if len(sub_year) > 0:
        traj = {int(k): int(v)
                for k, v in sub_year.astype(int).value_counts().sort_index().items()}
        trajectory_data[int(c)] = traj
        print(f"  C{c:02d}: {min(traj)}-{max(traj)} | "
              f"peak={max(traj, key=traj.get)}({max(traj.values())})")

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_json_safe(i) for i in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

with open(os.path.join(OUTPUT_DIR, "trajectories.json"), "w") as f:
    json.dump(make_json_safe(trajectory_data), f, indent=2)
print("Trajectories saved.")



In [ ]:
# ── STEP 16: Figures ──────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 16 — Generating figures")
print("=" * 60)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

stage_color = {
    "Emerging":   "#1a9850",
    "Growth":     "#f4a736",
    "Mature":     "#4393c3",
    "Saturation": "#d73027",
    "Unknown":    "#aaaaaa"
}

# Figure 1: Annual distribution
if len(annual) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(annual.index, annual.values, color="#2166ac",
           edgecolor="white", linewidth=0.4)
    ax.set_xlabel("Year", fontsize=12)
    ax.set_ylabel("Number of Grants", fontsize=12)
    ax.set_title("Figure 1: Annual Distribution of Supply Chain R&D Grants",
                 fontsize=13)
    peak_yr  = annual.idxmax()
    peak_val = annual.max()
    ax.annotate(str(peak_val), (peak_yr, peak_val),
                textcoords="offset points", xytext=(0, 5),
                ha="center", fontsize=9, color="red", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figure1_annual.png"), dpi=150)
    plt.close()
    print("Figure 1 saved.")

# Figure 2: Portfolio balance
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(stage_order))
w = 0.35
actual_vals = [actual_grant_shares.get(s, 0) for s in stage_order]
ideal_vals  = [ideal_shares[s] for s in stage_order]
bars1 = ax.bar(x - w/2, actual_vals, w, label="Actual",
               color=[stage_color.get(s, "#888") for s in stage_order], alpha=0.9)
ax.bar(x + w/2, ideal_vals, w, label="Ideal (Benchmark)",
       color="gray", alpha=0.5, hatch="//")
ax.set_xticks(x)
ax.set_xticklabels(stage_order, fontsize=11)
ax.set_ylabel("Share of Total Grants", fontsize=12)
ax.set_title("Figure 2: Portfolio Balance — Actual vs. Benchmark", fontsize=13)
ax.legend(fontsize=11)
for bar in bars1:
    ax.annotate(f"{bar.get_height():.2f}",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure2_portfolio_balance.png"), dpi=150)
plt.close()
print("Figure 2 saved.")

# Figure 3: Temporal trajectories
if len(trajectory_data) > 0:
    ncols = 3
    nrows = -(-BEST_K // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 3.5))
    axes = axes.flatten()
    for idx, c in enumerate(sorted(trajectory_data.keys())):
        traj  = trajectory_data[c]
        row   = df_lifecycle[df_lifecycle["cluster"] == c]
        stage = row["lifecycle_stage"].values[0] if len(row) > 0 else "Unknown"
        color = stage_color.get(stage, "#888")
        years  = sorted(traj.keys())
        counts = [traj[y] for y in years]
        axes[idx].bar(years, counts, color=color, edgecolor="white", linewidth=0.3)
        axes[idx].set_title(f"C{c:02d} — {stage}", fontsize=9)
        axes[idx].tick_params(axis="x", labelsize=7, rotation=45)
        axes[idx].tick_params(axis="y", labelsize=7)
    for i in range(idx + 1, len(axes)):
        axes[i].set_visible(False)
    fig.suptitle("Figure 3: Temporal Funding Trajectories by Cluster",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figure3_trajectories.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
    print("Figure 3 saved.")

# Figure 4: Agency-stage heatmap
if 'agency_stage_matrix' in locals() and agency_stage_matrix is not None and len(agency_stage_matrix) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(agency_stage_matrix, annot=True, fmt="d", cmap="YlGnBu",
                linewidths=0.5, ax=ax)
    ax.set_title("Figure 4: Agency-Stage Allocation Matrix", fontsize=13)
    ax.set_xlabel("Lifecycle Stage", fontsize=11)
    ax.set_ylabel("Funding Agency", fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figure4_agency_stage.png"), dpi=150)
    plt.close()
    print("Figure 4 saved.")



In [ ]:
# ── STEP 17: Expert Validation Template ──────────────────────
print("\n" + "=" * 60)
print("STEP 17 — Generating expert validation template")
print("=" * 60)

validation_samples = []
for c in sorted(df_raw["cluster"].unique()):
    sub = df_raw[df_raw["cluster"] == c]
    sample = sub.sample(n=min(5, len(sub)), random_state=42)
    for idx, row in sample.iterrows():
        validation_samples.append({
            "cluster_id": int(c),
            "grant_id": safe_get(row, "grant_id"),
            "title": safe_get(row, "title"),
            "abstract_snippet": safe_get(row, "abstract")[:200] + "...",
            "expert_rating": "",  # To be filled
            "relevance_comment": ""  # To be filled
        })

validation_df = pd.DataFrame(validation_samples)
validation_path = os.path.join(OUTPUT_DIR, "expert_validation_template.csv")
validation_df.to_csv(validation_path, index=False)
print(f"Expert validation template saved: {validation_path}")
print(f"Total samples for validation: {len(validation_samples)}")



In [ ]:
# ── STEP 18: Full Output Report ──────────────────────────────
print("\n" + "=" * 60)
print("STEP 18 — Compiling output report")
print("=" * 60)

lines = []
SEP = "=" * 60

lines += [SEP,
          "SUPPLY CHAIN R&D PORTFOLIO ANALYSIS — CLEAN CSV INPUT",
          SEP]

lines += ["\n[DATASET SUMMARY]",
          f"Total input CSV records: {len(df_raw)}",
          f"Input file: {CSV_PATH}",
          "No additional records were removed, normalized, or resampled.",
          f"Year range: {int(year_valid.min()) if len(year_valid)>0 else 'N/A'} "
          f"to {int(year_valid.max()) if len(year_valid)>0 else 'N/A'}",
          f"Records with valid year: {len(year_valid)}",
          f"Device: {DEVICE} | Model: {MODEL_NAME}",
          f"Clusters selected (k): {BEST_K}"]

if len(annual) > 0:
    lines += ["\n[ANNUAL GRANT DISTRIBUTION]"]
    for yr, cnt in annual.items():
        lines.append(f"  {int(yr)}: {int(cnt)}")

lines += ["\n[CLUSTERING EVALUATION TABLE]",
          df_eval.to_string(index=False),
          f"\nSelected k = {BEST_K}",
          f"  Silhouette:        {best_row['silhouette']:.4f}",
          f"  Davies-Bouldin:    {best_row['davies_bouldin']:.4f}",
          f"  Calinski-Harabasz: {best_row['calinski_harabasz']:.1f}",
          f"  Balance Ratio:     {best_row['balance_ratio']:.4f}",
          f"  Composite Score:   {best_row['score']:.4f}"]

lines += ["\n[CLUSTER SIZE DISTRIBUTION]"]
for c, cnt in cluster_sizes.items():
    lines.append(f"  C{int(c):02d}: {int(cnt)} grants ({int(cnt)/len(df_raw)*100:.1f}%)")

lines += ["\n[LIFECYCLE CLASSIFICATION TABLE]",
          df_lifecycle.to_string(index=False)]

lines += ["\n[LIFECYCLE STAGE DISTRIBUTION]"]
for stage in stage_order:
    cnt     = int(stage_counts.get(stage, 0))
    grant_n = int(df_lifecycle[df_lifecycle["lifecycle_stage"] == stage]["n_grants"].sum())
    lines.append(f"  {stage:12s}: {cnt} clusters | {grant_n} grants "
                 f"({grant_n/len(df_raw)*100:.1f}%)")

lines += ["\n[PORTFOLIO BALANCE ANALYSIS]",
          f"  HHI: {HHI} ({hhi_label})",
          f"  Portfolio Balance Score: {balance_score:.3f}",
          "\n  Actual vs. Ideal:"]
for s in stage_order:
    a = actual_grant_shares.get(s, 0)
    i = ideal_shares[s]
    d = ("over-invested" if (a-i) > 0.05
         else "under-invested" if (a-i) < -0.05
         else "on-target")
    lines.append(f"    {s:12s}: actual={a:.3f} | ideal={i:.3f} | gap={a-i:+.3f} | {d}")

if 'agency_stage_matrix' in locals() and agency_stage_matrix is not None:
    lines += ["\n[AGENCY-STAGE ALLOCATION MATRIX]",
              agency_stage_matrix.to_string()]

if agency_col and len(agency_df) > 0:
    lines += ["\n[TOP FUNDING AGENCIES]"]
    for _, row in agency_df.iterrows():
        lines.append(f"  {row['agency']}: {int(row['grant_count'])} grants ({row['pct']}%)")

lines += ["\n[TF-IDF KEYWORDS PER CLUSTER (top 15)]"]
for c in sorted(cluster_keywords.keys()):
    row   = df_lifecycle[df_lifecycle["cluster"] == c]
    stage = row["lifecycle_stage"].values[0] if len(row) > 0 else "?"
    kw_str = ", ".join(f"{w}({s:.3f})" for w, s in cluster_keywords[c])
    lines.append(f"  C{int(c):02d} [{stage}]: {kw_str}")

lines += ["\n[TEMPORAL TRAJECTORIES]"]
for c in sorted(trajectory_data.keys()):
    traj   = trajectory_data[c]
    peak_y = max(traj, key=traj.get)
    rec_avg = sum(traj.get(y, 0) for y in RECENT_WINDOW) / len(RECENT_WINDOW)
    lines.append(f"  C{int(c):02d}: years {min(traj)}-{max(traj)} | "
                 f"peak={peak_y}({traj[peak_y]}) | recent_avg={rec_avg:.1f}")

lines += ["\n[FIGURES GENERATED]",
          "  figure1_annual.png",
          "  figure2_portfolio_balance.png",
          "  figure3_trajectories.png",
          "  figure4_agency_stage.png"]

lines += ["\n[VALIDATION ASSETS]",
          f"  expert_validation_template.csv ({len(validation_samples)} samples)"]

lines += ["\n" + SEP, "END OF REPORT", SEP]

report = "\n".join(lines)
report_path = os.path.join(OUTPUT_DIR, "pipeline_output_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print(f"\nFull report saved: {report_path}")



In [ ]:
# ── STEP 19: Final File Listing ───────────────────────────────
print("\n" + "=" * 60)
print("STEP 19 — Output file listing")
print("=" * 60)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f"  {fname:<50} {os.path.getsize(fpath)/1024:>8.1f} KB")

print("\n" + "=" * 60)
print("PIPELINE COMPLETE — CLEAN CSV INPUT")
print(f"Input corpus retained: {len(df_raw)} grants")
print("=" * 60)